# GROUP 11 - SET 11

In [129]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [130]:
import pandas as pd
import numpy as np
import os
import glob
from datetime import datetime

In [131]:
# ==============================
# 2. CONFIGURATION
# ==============================

RUNS_DIRECTORY = "/content/drive/MyDrive/Colab Notebooks/set11/set11(dataset)"
qrels_file = "/content/drive/MyDrive/Colab Notebooks/set11/qrels.trec8.adhoc"
OUTPUT_FOLDER = "/content/drive/MyDrive/Colab Notebooks"

FILE_PATTERN = "*"
DOC_THRESHOLD = 1000


## 1.0 Data Cleaning

In [132]:
def read_clean_trec(file_path):
    column_names = ["query_id", "Q0", "doc_id", "rank", "score", "system_name"]

    try:
        df = pd.read_csv(
            file_path,
            sep=r'\s+',
            engine='python',
            names=column_names,
            dtype=str
        )
    except Exception as e:
        print(f"ERROR reading {file_path}: {e}")
        return pd.DataFrame()

    df["rank"] = pd.to_numeric(df["rank"], errors="coerce")
    df["score"] = pd.to_numeric(df["score"], errors="coerce")

    df = df.drop(columns=["Q0"])

    df = df.dropna(
        subset=["query_id", "doc_id", "rank", "score", "system_name"]
    ).copy()

    # Rebuild rank based on score descending
    df = df.sort_values(
        ["query_id", "score"],
        ascending=[True, False]
    ).copy()

    df["rank"] = df.groupby("query_id").cumcount() + 1
    df["rank"] = df["rank"].astype(int)

    return df

In [133]:
# ==============================
# 4. LOAD ALL RUN FILES
# ==============================

def load_all_runs(directory, pattern):
    file_paths = glob.glob(os.path.join(directory, pattern))

    if not file_paths:
        raise ValueError("No files found in the specified directory.")

    print(f"Found {len(file_paths)} files. Loading...")

    dataframes = []

    for file_path in file_paths:
        df = read_clean_trec(file_path)

        if not df.empty:
            dataframes.append(df)
            print(f"Loaded: {os.path.basename(file_path)} | rows: {len(df)}")
        else:
            print(f"Skipped empty/error file: {os.path.basename(file_path)}")

    clean_data = pd.concat(dataframes, ignore_index=True)

    clean_data = clean_data.sort_values(
        by=["system_name", "query_id", "rank"],
        ascending=[True, True, True]
    ).copy()

    return clean_data

In [134]:
# ==============================
# 5. RUN BASIC CLEANING
# ==============================

print(f"Attempting to load files from: {RUNS_DIRECTORY}")
if not os.path.exists(RUNS_DIRECTORY):
    print(f"Error: The directory {RUNS_DIRECTORY} does not exist.")
else:
    files_in_dir = os.listdir(RUNS_DIRECTORY)
    print(f"Files found in directory {RUNS_DIRECTORY}: {files_in_dir}")
    if not files_in_dir:
        print(f"Warning: The directory {RUNS_DIRECTORY} is empty.")

clean_data = load_all_runs(RUNS_DIRECTORY, FILE_PATTERN)

print("\nTotal cleaned rows:", len(clean_data))
display(clean_data.head())

Attempting to load files from: /content/drive/MyDrive/Colab Notebooks/set11/set11(dataset)
Files found in directory /content/drive/MyDrive/Colab Notebooks/set11/set11(dataset): ['input.Dm8TFidf', 'input.apl8c621', 'input.ibms99b', 'input.Mer8Adtd2', 'input.ok8asxc', 'input.pir9Attd', 'input.INQ601', 'input.GE8ATDN2', 'input.acsys8alo', 'input.nttd8ale', 'input.CL99XT', 'input.att99atde', 'input.fub99a', 'input.acsys8alo2', 'input.apl8ctd']
Found 15 files. Loading...
Loaded: input.Dm8TFidf | rows: 50000
Loaded: input.apl8c621 | rows: 50000
Loaded: input.ibms99b | rows: 50000
Loaded: input.Mer8Adtd2 | rows: 50000
Loaded: input.ok8asxc | rows: 50000
Loaded: input.pir9Attd | rows: 50000
Loaded: input.INQ601 | rows: 50000
Loaded: input.GE8ATDN2 | rows: 50000
Loaded: input.acsys8alo | rows: 50000
Loaded: input.nttd8ale | rows: 50000
Loaded: input.CL99XT | rows: 50000
Loaded: input.att99atde | rows: 50000
Loaded: input.fub99a | rows: 50000
Loaded: input.acsys8alo2 | rows: 50000
Loaded: input.

,query_id,doc_id,rank,score,system_name
500000,401,FT932-15086,1,11173.180,CL99XT
500001,401,FT944-6909,2,10887.392,CL99XT
500002,401,FT943-15609,3,10854.055,CL99XT
500003,401,FT924-7265,4,10381.393,CL99XT
500004,401,FBIS4-67877,5,562.130,CL99XT


In [135]:
# ==============================
# 6. BASIC VALIDATION
# ==============================

print("Data types:")
print(clean_data.dtypes)

print("\nMissing values:")
print(clean_data.isna().sum())

print("\nRank start check:")
print(clean_data.groupby(["system_name", "query_id"])["rank"].min().value_counts())

Data types:
query_id        object
doc_id          object
rank             int64
score          float64
system_name     object
dtype: object

Missing values:
query_id       0
doc_id         0
rank           0
score          0
system_name    0
dtype: int64

Rank start check:
rank
1    750
Name: count, dtype: int64


In [136]:
# ==============================
# 7. CHECK DUPLICATES
# ==============================

duplicates = clean_data.duplicated(
    subset=["system_name", "query_id", "doc_id"]
)

if duplicates.any():
    print("WARNING: Duplicate documents detected.")
    display(clean_data[duplicates].head())
else:
    print("OK: No duplicate documents found.")

OK: No duplicate documents found.


In [137]:
# ==============================
# 8. CHECK DOCUMENT COUNT PER ENTRY
# ==============================

query_doc_counts = (
    clean_data.groupby(["system_name", "query_id"])
    .size()
    .reset_index(name="doc_count")
)

display(query_doc_counts.head())

insufficient_queries = query_doc_counts[query_doc_counts["doc_count"] < DOC_THRESHOLD]

if not insufficient_queries.empty:
    print("WARNING: Some system-query pairs have fewer than 1000 documents.")
    display(insufficient_queries)
else:
    print("OK: All system-query pairs have enough documents.")

,system_name,query_id,doc_count
0,CL99XT,401,1000
1,CL99XT,402,1000
2,CL99XT,403,1000
3,CL99XT,404,1000
4,CL99XT,405,1000


OK: All system-query pairs have enough documents.


In [138]:
# ==============================
# 10. LOAD QRELS FILE
# ==============================

qrels_df = pd.read_csv(
    qrels_file,
    sep=r'\s+',
    header=None,
    names=["query_id", "ignore", "doc_id", "relevance"]
)

qrels_df.drop(columns=["ignore"], inplace=True)

print("Qrels loaded:", len(qrels_df), "rows")
display(qrels_df.head())

Qrels loaded: 86830 rows


,query_id,doc_id,relevance
0,401,FBIS3-10009,0
1,401,FBIS3-10059,0
2,401,FBIS3-10142,0
3,401,FBIS3-1026,0
4,401,FBIS3-10502,0


In [139]:
# ==============================
# 11. MATCH DATA TYPES BEFORE MERGING LATER
# ==============================

clean_data["query_id"] = clean_data["query_id"].astype(str)
clean_data["doc_id"] = clean_data["doc_id"].astype(str)

qrels_df["query_id"] = qrels_df["query_id"].astype(str)
qrels_df["doc_id"] = qrels_df["doc_id"].astype(str)
qrels_df["relevance"] = pd.to_numeric(qrels_df["relevance"], errors="coerce").fillna(0).astype(int)

In [140]:
# ==============================
# 12. EXPORT BASIC CLEAN DATA
# ==============================

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

clean_file = os.path.join(OUTPUT_FOLDER, f"clean_runs_{timestamp}.csv")

clean_data.to_csv(clean_file, index=False)

print("Exported clean data:")
print(clean_file)

Exported clean data:
/content/drive/MyDrive/Colab Notebooks/clean_runs_20260502_062950.csv


### 1.1 Preparation for Analysis

In [141]:
## merge
merged_df = pd.merge(
    clean_data,
    qrels_df,
    on=['query_id', 'doc_id'],
    how='left'
)

merged_df['relevance'] = merged_df['relevance'].fillna(0).astype(int)

print(merged_df.head())

  query_id       doc_id  rank      score system_name  relevance
0      401  FT932-15086     1  11173.180      CL99XT          1
1      401   FT944-6909     2  10887.392      CL99XT          1
2      401  FT943-15609     3  10854.055      CL99XT          0
3      401   FT924-7265     4  10381.393      CL99XT          0
4      401  FBIS4-67877     5    562.130      CL99XT          0


In [142]:
## handle missing relevance
## document not in qrel = not relevant

print("Relevance distribution:")
print(merged_df['relevance'].value_counts())

print("\nMissing relevance:")
print(merged_df['relevance'].isna().sum())

print("\nCheck at least one relevant per query:")
print(
    merged_df.groupby(['system_name','query_id'])['relevance'].sum().head()
)
merged_df['relevance'].isna().sum()


Relevance distribution:
relevance
0    704123
1     45877
Name: count, dtype: int64

Missing relevance:
0

Check at least one relevant per query:
system_name  query_id
CL99XT       401         172
             402          64
             403          21
             404         112
             405          35
Name: relevance, dtype: int64


np.int64(0)

In [143]:
## sort data properly
merged_df = merged_df.sort_values(
    ['system_name', 'query_id', 'rank']
).copy()

In [144]:
## quick validation
print("Check relevance distribution:")
print(merged_df['relevance'].value_counts())

print("\nCheck one query:")
display(merged_df[merged_df['query_id'] == '401'].head(10))

Check relevance distribution:
relevance
0    704123
1     45877
Name: count, dtype: int64

Check one query:


,query_id,doc_id,rank,score,system_name,relevance
0,401,FT932-15086,1,11173.180,CL99XT,1
1,401,FT944-6909,2,10887.392,CL99XT,1
2,401,FT943-15609,3,10854.055,CL99XT,0
3,401,FT924-7265,4,10381.393,CL99XT,0
4,401,FBIS4-67877,5,562.130,CL99XT,0
5,401,FBIS4-67533,6,542.709,CL99XT,0
6,401,FT934-9441,7,304.628,CL99XT,0
7,401,FT924-11400,8,286.923,CL99XT,1
8,401,FBIS3-18410,9,271.099,CL99XT,0
9,401,FT941-16483,10,264.945,CL99XT,0


## 2.0 Data Analysis


#### 2.1 Precision

##### Precision@10

In [145]:
## STEP 1: Precision@10 (per-topic)

def precision_at_10(group):
    # sort by score descending (as per your choice)
    group_sorted = group.sort_values(by='score', ascending=False).head(10)

    # compute precision@10
    return group_sorted['relevance'].sum() / 10

# compute per-topic P@10
p10_results = (
    merged_df
    .groupby(['system_name', 'query_id'])
    .apply(precision_at_10)
    .reset_index(name='P@10')
)

print("Per-topic Precision@10:")
display(p10_results.sample(15))

Per-topic Precision@10:


/tmp/ipykernel_12258/372182422.py:14: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(precision_at_10)


,system_name,query_id,P@10
644,nttd8ale,445,0.2
210,Mer8Adtd2,411,0.5
36,CL99XT,437,0.4
156,INQ601,407,0.7
141,GE8ATDN2,442,0.0
161,INQ601,412,0.6
569,ibms99b,420,0.8
68,Dm8TFidf,419,0.2
323,acsys8alo2,424,0.4
41,CL99XT,442,0.9


In [146]:
## STEP 2: Precision@10 (per-system)

avg_p10 = (
    p10_results
    .groupby('system_name')['P@10']
    .mean()
    .reset_index(name='Mean_P@10')
)

print("\nOverall Precision@10 per system:")

for _, row in avg_p10.iterrows():
    print(f"System: {row['system_name']} - Mean Precision@10: {row['Mean_P@10']:.4f}")


Overall Precision@10 per system:
System: CL99XT - Mean Precision@10: 0.6920
System: Dm8TFidf - Mean Precision@10: 0.3440
System: GE8ATDN2 - Mean Precision@10: 0.5120
System: INQ601 - Mean Precision@10: 0.4360
System: Mer8Adtd2 - Mean Precision@10: 0.4440
System: acsys8alo - Mean Precision@10: 0.5300
System: acsys8alo2 - Mean Precision@10: 0.4740
System: apl8c621 - Mean Precision@10: 0.5040
System: apl8ctd - Mean Precision@10: 0.4500
System: att99atde - Mean Precision@10: 0.5480
System: fub99a - Mean Precision@10: 0.5300
System: ibms99b - Mean Precision@10: 0.4600
System: nttd8ale - Mean Precision@10: 0.4940
System: ok8asxc - Mean Precision@10: 0.4880
System: pir9Attd - Mean Precision@10: 0.5080


##### Precision at k

In [148]:
k_values = [1, 10, 50, 500, 1000]

def precision_at_k(group, k):
    # Sort by score descending and take top k
    group_sorted = group.sort_values(by='score', ascending=False).head(k)
    # Safely handle graded relevance (>0) and divide by k
    relevant_count = (group_sorted['relevance'] > 0).sum()
    return relevant_count / k

precision_results = {}
avg_precision_results = {}
precision_matrix_results = {}

# Ensure the output folder exists
os.makedirs(f"{OUTPUT_FOLDER}/PRECISION", exist_ok=True)

for k in k_values:
    # 1. Per system-per query Precision@k
    result = (
        merged_df
        .groupby(['system_name', 'query_id'])
        .apply(lambda x: precision_at_k(x, k))
        .reset_index(name=f'P@{k}')
    )
    precision_results[k] = result

    # 2. Average Precision@k per system (for summary table later)
    avg_result = (
        result
        .groupby('system_name')[f'P@{k}']
        .mean()
        .reset_index(name=f'Mean_P@{k}')
    )
    avg_precision_results[k] = avg_result

    # =========================================================
    # 3. FORMAT INTO PIVOT MATRIX (Like Image)
    # =========================================================

    # Pivot: Rows = query_id, Columns = system_name
    matrix_df = result.pivot(index='query_id', columns='system_name', values=f'P@{k}').reset_index()

    # Get just the system columns
    system_cols = [col for col in matrix_df.columns if col != 'query_id']

    # Add 'Overall' column (row-wise mean)
    matrix_df['Overall'] = matrix_df[system_cols].mean(axis=1)

    # Calculate 'Overall' row (column-wise mean)
    overall_row = matrix_df[system_cols].mean().to_dict()
    overall_row['query_id'] = 'Overall'
    overall_row['Overall'] = matrix_df['Overall'].mean()

    # Append the Overall row to the bottom
    matrix_df = pd.concat([matrix_df, pd.DataFrame([overall_row])], ignore_index=True)

    # Save this specific matrix to Google Drive
    file_path = f"{OUTPUT_FOLDER}/PRECISION/precision_at_{k}_matrix.csv"
    matrix_df.to_csv(file_path, index=False)
    print(f"Saved Matrix: {file_path}")

    # Store for display
    precision_matrix_results[k] = matrix_df

/tmp/ipykernel_12258/3932087784.py:22: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: precision_at_k(x, k))


Saved Matrix: /content/drive/MyDrive/Colab Notebooks/PRECISION/precision_at_1_matrix.csv


/tmp/ipykernel_12258/3932087784.py:22: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: precision_at_k(x, k))


Saved Matrix: /content/drive/MyDrive/Colab Notebooks/PRECISION/precision_at_10_matrix.csv


/tmp/ipykernel_12258/3932087784.py:22: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: precision_at_k(x, k))


Saved Matrix: /content/drive/MyDrive/Colab Notebooks/PRECISION/precision_at_50_matrix.csv


/tmp/ipykernel_12258/3932087784.py:22: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: precision_at_k(x, k))


Saved Matrix: /content/drive/MyDrive/Colab Notebooks/PRECISION/precision_at_500_matrix.csv
Saved Matrix: /content/drive/MyDrive/Colab Notebooks/PRECISION/precision_at_1000_matrix.csv


/tmp/ipykernel_12258/3932087784.py:22: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: precision_at_k(x, k))


In [153]:
# DISPLAY AND SUMMARIZE RESULTS

# 1. Display the pivot matrix exactly like the picture (using k=10 as an example)
print("\n========== MATRIX FORMAT (Precision@10) ==========")
display(precision_matrix_results[10])

# 2. Combine Overall Precision scores into your final summary table
final_precision = avg_precision_results[k_values[0]]

for k in k_values[1:]:
    final_precision = pd.merge(
        final_precision,
        avg_precision_results[k],
        on='system_name'
    )

print("\n========== SUMMARY OVERALL PRECISION SCORES ==========")
display(final_precision.sample(15))

# 3. Save the summary file
summary_path = f"{OUTPUT_FOLDER}/PRECISION/overall_precision_all_k.csv"
final_precision.to_csv(summary_path, index=False)
print(f"\nSaved overall precision summary to: {summary_path}")


========== MATRIX FORMAT (Precision@10) ==========


,query_id,CL99XT,Dm8TFidf,GE8ATDN2,INQ601,Mer8Adtd2,acsys8alo,acsys8alo2,apl8c621,apl8ctd,att99atde,fub99a,ibms99b,nttd8ale,ok8asxc,pir9Attd,Overall
0,401,0.300,0.100,0.800,0.100,0.200,0.80,0.600,0.900,0.00,0.100,0.00,0.00,0.000,0.000,0.000,0.260000
1,402,0.700,0.600,0.800,0.300,0.600,0.90,0.800,0.900,0.80,0.800,0.70,0.20,0.700,0.700,0.300,0.653333
2,403,0.400,1.000,1.000,0.800,0.700,1.00,1.000,0.900,1.00,1.000,1.00,0.90,1.000,1.000,1.000,0.913333
3,404,0.600,0.500,0.500,0.200,0.400,0.30,0.400,0.400,0.40,0.400,0.50,0.20,0.200,0.300,0.500,0.386667
4,405,0.600,0.400,0.500,0.100,0.300,0.50,0.200,0.600,0.60,0.300,0.50,0.40,0.400,0.400,0.500,0.420000
5,406,0.600,0.200,0.300,0.300,0.400,0.60,0.400,0.600,0.60,0.600,0.50,0.50,0.400,0.600,0.500,0.473333
6,407,0.900,0.400,0.900,0.700,0.800,0.70,0.200,0.900,0.80,0.900,0.90,0.80,0.800,0.800,0.700,0.746667
7,408,0.800,0.500,0.300,0.200,0.400,0.00,0.200,0.200,0.30,0.300,0.30,0.30,0.500,0.400,0.400,0.340000
8,409,0.500,0.400,0.400,0.300,0.300,0.40,0.400,0.300,0.30,0.500,0.30,0.20,0.300,0.100,0.200,0.326667
9,410,1.000,1.000,1.000,1.000,1.000,1.00,1.000,1.000,1.00,1.000,1.00,1.00,1.000,1.000,1.000,1.000000



========== SUMMARY OVERALL PRECISION SCORES ==========


,system_name,Mean_P@1,Mean_P@10,Mean_P@50,Mean_P@500,Mean_P@1000
3,INQ601,0.44,0.436,0.2896,0.08716,0.05402
5,acsys8alo,0.58,0.530,0.3452,0.11024,0.06624
8,apl8ctd,0.56,0.450,0.2984,0.10260,0.06234
11,ibms99b,0.56,0.460,0.3192,0.10252,0.06302
9,att99atde,0.62,0.548,0.3572,0.11004,0.06512
12,nttd8ale,0.66,0.494,0.3276,0.10376,0.06240
1,Dm8TFidf,0.56,0.344,0.1992,0.06044,0.03844
6,acsys8alo2,0.60,0.474,0.3168,0.10096,0.06160
14,pir9Attd,0.58,0.508,0.3452,0.11164,0.06684
4,Mer8Adtd2,0.60,0.444,0.2788,0.09000,0.05662



Saved overall precision summary to: /content/drive/MyDrive/Colab Notebooks/PRECISION/overall_precision_all_k.csv


### 2.2 MAP

In [150]:
# MAP@K CALCULATION

# 1. Total relevant documents per query (from qrels)
total_rel_per_query = (
    qrels_df[qrels_df['relevance'] > 0]
    .groupby('query_id')
    .size()
    .to_dict()
)


In [151]:
# 2. Function to calculate AP at a specific depth 'k'
def average_precision_at_k(group, k):
    # Get the query_id to look up the true total relevant docs
    qid = group['query_id'].iloc[0]
    total_relevant = total_rel_per_query.get(qid, 0)

    # If the dataset has no relevant docs for this query, score is 0
    if total_relevant == 0:
        return 0.0

    # Sort by score descending and take top k
    group_sorted = group.sort_values(by='score', ascending=False).head(k)

    relevant_count = 0
    precision_sum = 0

    # Fast loop using your itertuples method
    for i, row in enumerate(group_sorted.itertuples(), start=1):
        if row.relevance > 0:
            relevant_count += 1
            precision_sum += relevant_count / i

    # Divide by the TRUE total relevant, not the local amount!
    return precision_sum / total_relevant


ap_results = {}
map_results = {}
map_matrix_results = {}

# Create a folder to save the CSVs (optional, but keeps things clean)
os.makedirs(f"{OUTPUT_FOLDER}/MAP_Results", exist_ok=True)

for k in k_values:
    # 1. Calculate Per system-per query AP@k
    result = (
        merged_df
        .groupby(['system_name', 'query_id'])
        .apply(lambda x: average_precision_at_k(x, k))
        .reset_index(name=f'AP@{k}')
    )
    ap_results[k] = result

    # 2. Calculate MAP@k per system (for the summary table later)
    avg_map = (
        result
        .groupby('system_name')[f'AP@{k}']
        .mean()
        .reset_index(name=f'MAP@{k}')
    )
    map_results[k] = avg_map

    # Pivot the data: Rows = Topics (query_id), Columns = Systems
    matrix_df = result.pivot(index='query_id', columns='system_name', values=f'AP@{k}').reset_index()

    # Get a list of just the system columns
    system_cols = [col for col in matrix_df.columns if col != 'query_id']

    # Add 'Overall' column on the far right (Average of all systems for that topic)
    matrix_df['Overall'] = matrix_df[system_cols].mean(axis=1)

    # Calculate the 'Overall' row for the bottom (This is the MAP score for each system!)
    overall_row = matrix_df[system_cols].mean().to_dict()
    overall_row['query_id'] = 'Overall'
    overall_row['Overall'] = matrix_df['Overall'].mean() # Average of the averages

    # Append the Overall row to the bottom of the dataframe
    matrix_df = pd.concat([matrix_df, pd.DataFrame([overall_row])], ignore_index=True)

    # Save this beautiful matrix to a CSV file!
    file_path = f"{OUTPUT_FOLDER}/MAP_Results/MAP_at_{k}.csv"
    matrix_df.to_csv(file_path, index=False)
    print(f"Saved: {file_path}")

    # Store it in our dictionary so we can look at it in notebook
    map_matrix_results[k] = matrix_df

    print(f"Calculated and saved matrix for MAP@{k}")

/tmp/ipykernel_12258/3958251174.py:39: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: average_precision_at_k(x, k))


Saved: /content/drive/MyDrive/Colab Notebooks/MAP_Results/MAP_at_1.csv
Calculated and saved matrix for MAP@1


/tmp/ipykernel_12258/3958251174.py:39: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: average_precision_at_k(x, k))


Saved: /content/drive/MyDrive/Colab Notebooks/MAP_Results/MAP_at_10.csv
Calculated and saved matrix for MAP@10


/tmp/ipykernel_12258/3958251174.py:39: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: average_precision_at_k(x, k))


Saved: /content/drive/MyDrive/Colab Notebooks/MAP_Results/MAP_at_50.csv
Calculated and saved matrix for MAP@50


/tmp/ipykernel_12258/3958251174.py:39: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: average_precision_at_k(x, k))


Saved: /content/drive/MyDrive/Colab Notebooks/MAP_Results/MAP_at_500.csv
Calculated and saved matrix for MAP@500
Saved: /content/drive/MyDrive/Colab Notebooks/MAP_Results/MAP_at_1000.csv
Calculated and saved matrix for MAP@1000


/tmp/ipykernel_12258/3958251174.py:39: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: average_precision_at_k(x, k))


In [152]:
# DISPLAY THE RESULTS

# Show the matrix for k=10 as an example of what was saved to CSV
print("\n========== EXAMPLE MATRIX FORMAT (MAP@10) ==========")
display(map_matrix_results[10])

# Keep your excellent summary merging code!
# This creates a table comparing MAP@1, MAP@10, etc., side-by-side
final_map_summary = map_results[k_values[0]]
for k in k_values[1:]:
    final_map_summary = pd.merge(
        final_map_summary,
        map_results[k],
        on='system_name'
    )

print("\n========== SUMMARY OVERALL MAP SCORES ==========")
display(final_map_summary)

# Optionally save the summary table too
final_map_summary.to_csv(
    f"{OUTPUT_FOLDER}/MAP_Results/overall_MAP_summary_all_k.csv",
    index=False
)


========== EXAMPLE MATRIX FORMAT (MAP@10) ==========


,query_id,CL99XT,Dm8TFidf,GE8ATDN2,INQ601,Mer8Adtd2,acsys8alo,acsys8alo2,apl8c621,apl8ctd,att99atde,fub99a,ibms99b,nttd8ale,ok8asxc,pir9Attd,Overall
0,401,0.007917,0.000417,0.023348,0.000333,0.004444,0.018751,0.012417,0.029667,0.000000,0.000417,0.000000,0.000000,0.000000,0.000000,0.000000,0.006514
1,402,0.082937,0.065312,0.091597,0.017188,0.059137,0.112500,0.094722,0.106513,0.086443,0.095263,0.079568,0.020833,0.065729,0.066845,0.020238,0.070988
2,403,0.129365,0.476190,0.476190,0.341213,0.226587,0.476190,0.476190,0.423810,0.476190,0.476190,0.476190,0.428571,0.476190,0.476190,0.476190,0.420763
3,404,0.025156,0.019855,0.028672,0.002934,0.013190,0.009859,0.018310,0.025822,0.020736,0.009859,0.027582,0.005164,0.008803,0.008979,0.013777,0.015913
4,405,0.141667,0.065476,0.074196,0.002924,0.029449,0.083041,0.023684,0.127412,0.135673,0.049342,0.074687,0.044904,0.058406,0.033469,0.076754,0.068072
5,406,0.312546,0.089744,0.128205,0.186813,0.195360,0.327350,0.225275,0.253571,0.319658,0.379304,0.262821,0.299359,0.218315,0.338889,0.281197,0.254560
6,407,0.132353,0.058824,0.130882,0.099673,0.098693,0.068575,0.017974,0.132353,0.103560,0.132353,0.129248,0.106886,0.112868,0.117647,0.093627,0.102368
7,408,0.065042,0.035381,0.012107,0.008475,0.024247,0.000000,0.003578,0.011299,0.008898,0.017302,0.025424,0.023305,0.026500,0.018738,0.011461,0.019450
8,409,0.134686,0.124242,0.080087,0.113636,0.098485,0.087879,0.116883,0.107955,0.125000,0.142136,0.077652,0.060606,0.087121,0.045455,0.090909,0.099515
9,410,0.153846,0.153846,0.153846,0.153846,0.153846,0.153846,0.153846,0.153846,0.153846,0.153846,0.153846,0.153846,0.153846,0.153846,0.153846,0.153846



========== SUMMARY OVERALL MAP SCORES ==========


,system_name,MAP@1,MAP@10,MAP@50,MAP@500,MAP@1000
0,CL99XT,0.018643,0.130851,0.254820,0.365223,0.373035
1,Dm8TFidf,0.013362,0.073952,0.123222,0.158057,0.163042
2,GE8ATDN2,0.016207,0.088222,0.162404,0.249445,0.257970
3,INQ601,0.011858,0.080583,0.157453,0.224810,0.232544
4,Mer8Adtd2,0.014656,0.077695,0.146641,0.214132,0.223133
5,acsys8alo,0.014283,0.101239,0.191688,0.283580,0.293502
6,acsys8alo2,0.016356,0.095007,0.174114,0.254326,0.263668
7,apl8c621,0.017370,0.105636,0.200697,0.302168,0.312633
8,apl8ctd,0.016495,0.091294,0.182512,0.277154,0.285988
9,att99atde,0.014879,0.099052,0.201852,0.307403,0.316549


### 2.3 NDCG